In [1]:
# ============================================================
# Preparing the Oscar award data for the AWARD_Movie table
#
# Written by: Capleton

# ------------------------------------------------------------
# STEP 1: import pandas
# What: we load the pandas library.
# Why:  pandas lets us read, filter, sort and save tables
#       (DataFrames).
# ------------------------------------------------------------
import pandas as pd




In [7]:
# ------------------------------------------------------------
# STEP 2: read the file
# What: the CSV becomes a table in Python (10,889 rows).
# Why:  the original file is only read and stays unchanged,
#       so we can always go back to the full data.
# ------------------------------------------------------------
df = pd.read_csv(r"C:\Users\caple\OneDrive - HvA\Master\Data management\Data\oscarAward.csv")
df.head(10)



,year_film,year_ceremony,ceremony,category,name,film,winner
0,1927,1928,1,ACTOR,Richard Barthelmess,The Noose,False
1,1927,1928,1,ACTOR,Emil Jannings,The Last Command,True
2,1927,1928,1,ACTRESS,Louise Dresser,A Ship Comes In,False
3,1927,1928,1,ACTRESS,Janet Gaynor,7th Heaven,True
4,1927,1928,1,ACTRESS,Gloria Swanson,Sadie Thompson,False
5,1927,1928,1,ART DIRECTION,Rochus Gliese,Sunrise,False
6,1927,1928,1,ART DIRECTION,William Cameron Menzies,The Dove;,True
7,1927,1928,1,ART DIRECTION,Harry Oliver,7th Heaven,False
8,1927,1928,1,CINEMATOGRAPHY,George Barnes,The Devil Dancer;,False
9,1927,1928,1,CINEMATOGRAPHY,Charles Rosher,Sunrise,True


In [16]:

# STEP 3: keep only the winners with a film
# What: only rows where "winner" is True and "film" is not
#       empty are kept.
# Why:  our research looks at the effect of WINNING an Oscar
#       on box office. Awards without a film (e.g. honorary
#       awards for a career) do not belong to a movie.
# ------------------------------------------------------------
winners = df[(df["winner"] == True) & (df["film"].notna())]
 
print("\n=== STEP 3: only winners with a film ===")
print("Rows before:", len(df))
print("Rows after: ", len(winners))
print("Values left in 'winner':", winners["winner"].unique())


=== STEP 3: only winners with a film ===
Rows before: 10889
Rows after:  2172
Values left in 'winner': [ True]


In [13]:

# STEP 4: keep only the movie awards
# What: rows whose category starts with ACTOR, ACTRESS or
#       DIRECTING are removed. ~ means "not".
# Why:  those awards belong in AWARD_Actor and AWARD_Director.
#       AWARD_Movie only holds the other categories, like
#       Best Picture, Music and Cinematography.
#       We use startswith on purpose: searching for "DIRECT"
#       would also remove Art Direction and Dance Direction,
#       which ARE movie awards.

award_movie = winners[~winners["category"].str.startswith(("ACTOR", "ACTRESS", "DIRECTING"))].copy()
print(winners.head())

    year_film  year_ceremony  ceremony        category  \
1        1927           1928         1           ACTOR   
3        1927           1928         1         ACTRESS   
6        1927           1928         1   ART DIRECTION   
9        1927           1928         1  CINEMATOGRAPHY   
10       1927           1928         1  CINEMATOGRAPHY   

                       name              film  winner  
1             Emil Jannings  The Last Command    True  
3              Janet Gaynor        7th Heaven    True  
6   William Cameron Menzies         The Dove;    True  
9            Charles Rosher           Sunrise    True  
10              Karl Struss           Sunrise    True  


In [18]:
award_movie = winners[~winners["category"].str.startswith(("ACTOR", "ACTRESS", "DIRECTING"))].copy()
 

print("Rows before:", len(winners))
print("Rows after: ", len(award_movie))
print("Actor/director rows left:",
      award_movie["category"].str.startswith(("ACTOR", "ACTRESS", "DIRECTING")).sum())
print("Some categories that are kept:", list(award_movie["category"].unique()[:5]))
 

Rows before: 2172
Rows after:  1705
Actor/director rows left: 0
Some categories that are kept: ['ART DIRECTION', 'CINEMATOGRAPHY', 'ENGINEERING EFFECTS', 'OUTSTANDING PICTURE', 'UNIQUE AND ARTISTIC PICTURE']


In [19]:

# ------------------------------------------------------------
# STEP 5: match the column names to the ERD
# What: "name" becomes "name_nominee", "film" becomes
#       "film_title".
# Why:  this way the columns match the table in PostgreSQL.

print("Columns before:", list(award_movie.columns))
 
award_movie = award_movie.rename(columns={"name": "name_nominee", "film": "film_title"})
 
print("Columns after: ", list(award_movie.columns))

Columns before: ['year_film', 'year_ceremony', 'ceremony', 'category', 'name', 'film', 'winner']
Columns after:  ['year_film', 'year_ceremony', 'ceremony', 'category', 'name_nominee', 'film_title', 'winner']


In [20]:

# STEP 6: remove the ";" from the film titles
# What: rstrip(";") removes a ";" at the end of the title,
#       strip() removes spaces at the beginning and end.
#       "The Dove;" becomes "The Dove".
# Why:  a few old titles in the source data end with a ";",
#       which is a leftover and not part of the title.

print("Titles with ';' before:",
      award_movie.loc[award_movie["film_title"].str.contains(";"), "film_title"].tolist())
 
award_movie["film_title"] = award_movie["film_title"].str.rstrip(";").str.strip()
 
print("Titles with ';' after: ", award_movie["film_title"].str.contains(";").sum())

Titles with ';' before: ['The Dove;', 'Broadway Melody of 1936;']
Titles with ';' after:  0


In [21]:
# STEP 7: sort the table
# What: the rows are sorted by year_ceremony, then by category
#       and then by film_title.
# Why:  a clear, fixed order makes the table easy to read and
#       check, and gives the same award_id every time the
#       script runs.
# ------------------------------------------------------------
award_movie = award_movie.sort_values(["year_ceremony", "category", "film_title"])
 

print(award_movie[["year_ceremony", "category", "film_title"]].head())

    year_ceremony             category film_title
6            1928        ART DIRECTION   The Dove
9            1928       CINEMATOGRAPHY    Sunrise
10           1928       CINEMATOGRAPHY    Sunrise
17           1928  ENGINEERING EFFECTS      Wings
21           1928  OUTSTANDING PICTURE      Wings


In [22]:
# STEP 8: give every award a unique award_id
# What: reset_index renumbers the rows 0, 1, 2... Then every
#       award gets a number from 1 to 1,705.
# Why:  award_id is the primary key of AWARD_Movie. A primary
#       key must be unique and cannot be empty.
# ------------------------------------------------------------
award_movie = award_movie.reset_index(drop=True)
award_movie["award_id"] = range(1, len(award_movie) + 1)
 

print("First award_id:", award_movie["award_id"].min())
print("Last award_id: ", award_movie["award_id"].max())
print("All unique:    ", award_movie["award_id"].is_unique)
 

First award_id: 1
Last award_id:  1705
All unique:     True


In [24]:
# STEP 9: add an empty movie_id column
# What: the table gets a movie_id column without values.
# Why:  movie_id is in our ERD (foreign key to MOVIE), so the
#       column must exist. It stays empty for now and is
#       filled in later. Int64 keeps it a whole number.
# ------------------------------------------------------------
award_movie["movie_id"] = pd.Series(dtype="Int64")
 

print("Empty movie_id values:", award_movie["movie_id"].isna().sum(), "of", len(award_movie))
 

Empty movie_id values: 1705 of 1705


In [26]:
# STEP 10: keep only the ERD columns, in the ERD order
# What: only the columns from the ERD are kept.
# Why:  so the CSV fits the AWARD_Movie table in PostgreSQL.
#       "winner" is dropped (every row is a winner) and
#       "year_film" is dropped (it is always year_ceremony - 1).

award_movie = award_movie[["award_id", "movie_id", "year_ceremony", "ceremony", "category", "name_nominee", "film_title"]]
 

print(list(award_movie.columns))

['award_id', 'movie_id', 'year_ceremony', 'ceremony', 'category', 'name_nominee', 'film_title']


In [33]:
# STEP 11: save as a CSV file
# What: the table is saved as award_movie.csv in the same
#       folder as the original file.
# Why:  this file is imported into AWARD_Movie in pgAdmin.
#       A new file name, so the original is not overwritten.
#       index=False prevents an extra column with row numbers.

# STEP 11: save as a CSV file
# What: the table is saved as award_movie.csv in the same
#       folder as the original file.
# Why:  this file is imported into AWARD_Movie in pgAdmin.
#       A new file name, so the original is not overwritten.
#       index=False prevents an extra column with row numbers.

import os

folder = r"C:\Users\caple\OneDrive - HvA\Master\Data management\Data"
new_file = os.path.join(folder, "award_movie_clean.csv")

try:
    award_movie.to_csv(new_file, index=False)
    print("File saved as:", new_file)
    print("File exists:", os.path.exists(new_file))
except NameError:
    print("ERROR: award_movie does not exist yet. Run steps 1 to 10 first.")
except PermissionError:
    print("ERROR: the file is open in Excel. Close it and run this cell again.")
except OSError as error:
    print("ERROR: something is wrong with the folder or name:", error)

File saved as: C:\Users\caple\OneDrive - HvA\Master\Data management\Data\award_movie_clean.csv
File exists: True
